In [38]:
import numpy as np
import pandas as pd

The Negative (respectively Positive) category includes articles with a negative (respectively positive) tone and/or reporting violent events. Within these categories, a distinction is made between:
- Statements, i.e., purely verbal events
- Actions, i.e., performative events
- Violence, included only in the Negative category

In the bilateral version, the indicators are defined as follows:
- Shade: the share of articles belonging to a given category (for example, Negative) that mention two countries simultaneously, relative to all articles mentioning these two countries during a given month. Example: if, during a month, 20% of articles mentioning France and the United States have a negative tone, the Negative Shade between these two countries equals 0.2.
- Intensity: the share of articles mentioning two countries, relative to all articles mentioning the first country in the pair during the same month. Example: if articles mentioning France and the United States account for 10% of all articles mentioning France, the Intensity of the United States for France equals 0.1.

In [39]:
df = pd.read_csv("../Raw/IntenSE_2025_v1/IntenSe_2025_bilateral_v1.csv")
df

,Date,Actor1CountryCode,Actor2CountryCode,Category,NumEvents,NumArticles,Shade,Intensity
0,201503,AFG,AUS,Negative,1,1,0.5,0.003960
1,201503,AUS,AFG,Negative,1,1,0.5,0.004008
2,201503,AFG,AUS,Violence,1,1,0.5,0.003960
3,201503,AUS,AFG,Violence,1,1,0.5,0.004008
4,201503,AFG,AUS,DeclarationPositive,1,1,0.5,0.003960
...,...,...,...,...,...,...,...,...
1691035,202507,ZAF,USA,DeclarationPositive,1,1,0.2,0.121951
1691036,202507,USA,ZAF,Negative,3,3,0.6,0.002844
1691037,202507,ZAF,USA,Negative,3,3,0.6,0.121951
1691038,202507,USA,ZAF,Positive,2,2,0.4,0.002844


In [40]:
# Actor1 is the country that writes the news.
# Actor2 is the country that the news is about.
# Rename to iso3_i for origin and iso3_j for destination
df = df.rename(columns={"Actor1CountryCode": "iso3_i", "Actor2CountryCode": "iso3_j"})

In [41]:
df["Category"].value_counts()

Category
Negative               354234
Positive               333904
DeclarationPositive    297442
DeclarationNegative    277428
ActionNegative         158762
ActionPositive         143078
Violence               126192
Name: count, dtype: int64

In [42]:
# Now group by year
df["Date"] = df["Date"].astype(str)
df["year"] = df["Date"].str[:4]
df_yearly = df.groupby(["year", "iso3_i", "iso3_j"])[["NumEvents", "NumArticles"]].sum().reset_index()
df_yearly

,year,iso3_i,iso3_j,NumEvents,NumArticles
0,2015,AFG,ALB,8,12
1,2015,AFG,ARE,20,28
2,2015,AFG,ARM,28,28
3,2015,AFG,AUS,16,16
4,2015,AFG,AUT,16,20
...,...,...,...,...,...
71205,2025,ZAF,USA,106,106
71206,2025,ZMB,BGD,6,6
71207,2025,ZMB,CHN,12,12
71208,2025,ZMB,ETH,16,16


In [43]:
# Get the sum per destination and year
df_yearly["Sum_NumArticles"] = df_yearly.groupby(["year", "iso3_j"])["NumArticles"].transform("sum")
df_yearly["Sum_NumEvents"] = df_yearly.groupby(["year", "iso3_j"])["NumEvents"].transform("sum")

df_yearly

,year,iso3_i,iso3_j,NumEvents,NumArticles,Sum_NumArticles,Sum_NumEvents
0,2015,AFG,ALB,8,12,1604,1440
1,2015,AFG,ARE,20,28,8014,4922
2,2015,AFG,ARM,28,28,11292,9988
3,2015,AFG,AUS,16,16,8442,6102
4,2015,AFG,AUT,16,20,8738,6402
...,...,...,...,...,...,...,...
71205,2025,ZAF,USA,106,106,26264,22668
71206,2025,ZMB,BGD,6,6,1524,1440
71207,2025,ZMB,CHN,12,12,10112,9340
71208,2025,ZMB,ETH,16,16,466,458


In [44]:
# Calculate sum intensity over the year
df_yearly["Intensity_Articles"] = df_yearly["NumArticles"] / df_yearly["Sum_NumArticles"]
df_yearly

,year,iso3_i,iso3_j,NumEvents,NumArticles,Sum_NumArticles,Sum_NumEvents,Intensity_Articles
0,2015,AFG,ALB,8,12,1604,1440,0.007481
1,2015,AFG,ARE,20,28,8014,4922,0.003494
2,2015,AFG,ARM,28,28,11292,9988,0.002480
3,2015,AFG,AUS,16,16,8442,6102,0.001895
4,2015,AFG,AUT,16,20,8738,6402,0.002289
...,...,...,...,...,...,...,...,...
71205,2025,ZAF,USA,106,106,26264,22668,0.004036
71206,2025,ZMB,BGD,6,6,1524,1440,0.003937
71207,2025,ZMB,CHN,12,12,10112,9340,0.001187
71208,2025,ZMB,ETH,16,16,466,458,0.034335


In [45]:
# Log Articles and Events
df_yearly["Log_NumArticles"] = np.log1p(df_yearly["NumArticles"])
df_yearly["Log_NumEvents"] = np.log1p(df_yearly["NumEvents"])

In [46]:
df_yearly.sort_values("Intensity_Articles", ascending=False).head(10)

,year,iso3_i,iso3_j,NumEvents,NumArticles,Sum_NumArticles,Sum_NumEvents,Intensity_Articles,Log_NumArticles,Log_NumEvents
71099,2025,VEN,BRB,6,6,6,6,1.0,1.945910,1.945910
15329,2016,USA,FSM,10,14,14,10,1.0,2.708050,2.397895
63381,2024,CHN,GNB,6,6,6,6,1.0,1.945910,1.945910
70980,2025,USA,FSM,8,8,8,8,1.0,2.197225,2.197225
64269,2024,FRA,MDG,6,6,6,6,1.0,1.945910,1.945910
70998,2025,USA,JAM,10,10,10,10,1.0,2.397895,2.397895
39432,2019,USA,LCA,34,126,126,34,1.0,4.844187,3.555348
63410,2024,CHN,MAC,28,28,28,28,1.0,3.367296,3.367296
62366,2023,ZAF,LSO,6,6,6,6,1.0,1.945910,1.945910
64476,2024,GHA,TGO,14,14,14,14,1.0,2.708050,2.708050


In [47]:
# Inspect sweden
df_swe = df_yearly[(df_yearly["iso3_j"] == "SWE")]
df_swe

,year,iso3_i,iso3_j,NumEvents,NumArticles,Sum_NumArticles,Sum_NumEvents,Intensity_Articles,Log_NumArticles,Log_NumEvents
52,2015,AFG,SWE,26,46,3806,2850,0.012086,3.850148,3.295837
105,2015,ALB,SWE,14,14,3806,2850,0.003678,2.708050,2.708050
170,2015,ARE,SWE,34,54,3806,2850,0.014188,4.007333,3.555348
225,2015,ARG,SWE,50,50,3806,2850,0.013137,3.931826,3.931826
296,2015,ARM,SWE,22,22,3806,2850,0.005780,3.135494,3.135494
...,...,...,...,...,...,...,...,...,...,...
70819,2025,TUR,SWE,12,12,918,818,0.013072,2.564949,2.564949
70918,2025,UKR,SWE,112,124,918,818,0.135076,4.828314,4.727388
70937,2025,URY,SWE,8,8,918,818,0.008715,2.197225,2.197225
71050,2025,USA,SWE,46,46,918,818,0.050109,3.850148,3.850148


In [48]:
# Export to csv
df_yearly.to_csv("../Clean/IntenSE_yearly.csv", index=False)